# Ejercicio 1 — Feature Extraction binario: Caballos vs Humanos con MobileNetV2

**Curso:** IS056B — Ingeniería del Conocimiento | **Sesión 09-A:** Transfer Learning — Feature Extraction

**Objetivo:** Aplicar Feature Extraction a un problema de **clasificación binaria (2 clases)**: determinar si una imagen contiene un **caballo** o un **humano**.

**Dataset:** Horses or Humans — 1,283 imágenes sintéticas (300×300 RGB) de caballos y personas en distintas poses y fondos. Se usan 1,027 imágenes para entrenamiento/validación y 256 como conjunto de prueba.

**Modelo base:** MobileNetV2 pre-entrenado en ImageNet (ligero, ~2.2M parámetros en el backbone sin cabeza).

## Configuración del entorno

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
print("GPU detectada:", gpu_info.stdout.strip() or "No hay GPU - activa en Runtime > Change runtime type > T4 GPU")

In [ ]:
import os, zipfile
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

---
## 1.1. Descargar y explorar el dataset

Los archivos se descargan directamente (con enlace de respaldo) y se extraen en la carpeta `datos/`.

In [ ]:
DATA_DIR = os.path.abspath('datos')
os.makedirs(DATA_DIR, exist_ok=True)

def descargar(nombre, urls):
    ultimo_error = None
    for url in urls:
        try:
            return tf.keras.utils.get_file(nombre, origin=url, cache_dir=DATA_DIR,
                                           cache_subdir='descargas')
        except Exception as e:
            ultimo_error = e
            print(f"No se pudo descargar desde {url}: {e}")
    raise RuntimeError(f"No fue posible descargar {nombre}") from ultimo_error

def raiz_con_carpetas(base, carpetas):
    for r, dirs, _ in os.walk(base):
        if all(c in dirs for c in carpetas):
            return r
    raise FileNotFoundError(f"No se encontraron las carpetas {carpetas} en {base}")

zip_train = descargar('horse-or-human.zip', [
    'https://storage.googleapis.com/learning-datasets/horse-or-human.zip',
    'https://storage.googleapis.com/laurencemoroney-blog.appspot.com/horse-or-human.zip'])
zip_test = descargar('validation-horse-or-human.zip', [
    'https://storage.googleapis.com/learning-datasets/validation-horse-or-human.zip',
    'https://storage.googleapis.com/laurencemoroney-blog.appspot.com/validation-horse-or-human.zip'])

for ruta, destino in [(zip_train, 'train'), (zip_test, 'test')]:
    carpeta = os.path.join(DATA_DIR, destino)
    if not os.path.isdir(carpeta):
        with zipfile.ZipFile(ruta) as z:
            z.extractall(carpeta)

train_dir = raiz_con_carpetas(os.path.join(DATA_DIR, 'train'), ['horses', 'humans'])
test_dir  = raiz_con_carpetas(os.path.join(DATA_DIR, 'test'),  ['horses', 'humans'])
print("Carpeta de entrenamiento:", train_dir, os.listdir(train_dir))
print("Carpeta de prueba:       ", test_dir, os.listdir(test_dir))

In [ ]:
IMG_SIZE_1 = (224, 224)
BATCH_SIZE = 32
FOLDERS = ['horses', 'humans']          # 0 = caballo, 1 = humano
CLASS_NAMES_1 = ['Caballo', 'Humano']

common = dict(image_size=IMG_SIZE_1, batch_size=BATCH_SIZE,
              label_mode='binary', class_names=FOLDERS)

train_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset='training', seed=SEED, **common)
val_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset='validation', seed=SEED, **common)
test_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, shuffle=False, **common)

In [ ]:
# Distribución de clases por partición
def contar(ds):
    c = Counter(os.path.basename(os.path.dirname(p)) for p in ds.file_paths)
    return [c[f] for f in FOLDERS]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ds, name in zip(axes, [train_raw, val_raw, test_raw], ['Train', 'Validación', 'Test']):
    counts = contar(ds)
    bars = ax.bar(CLASS_NAMES_1, counts, color=['#8D6E63', '#42A5F5'], edgecolor='gray')
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width()/2, c + 3, str(c), ha='center', fontweight='bold')
    ax.set_title(name, fontweight='bold'); ax.set_ylabel('Cantidad')
plt.suptitle('Horses or Humans — Distribución de clases', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Muestras del dataset
imgs, labels = next(iter(train_raw))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs[i].numpy().astype('uint8'))
    idx = int(labels[i].numpy()[0])
    ax.set_title(CLASS_NAMES_1[idx], fontsize=12,
                 color='saddlebrown' if idx == 0 else 'dodgerblue')
    ax.axis('off')
plt.suptitle('Horses or Humans — Muestras del dataset', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 1.2. Preprocesamiento

MobileNetV2 espera imágenes de **224×224** con píxeles en **[-1, 1]**, que es lo que hace su `preprocess_input`. Como el problema es binario, la etiqueta es un valor 0/1 y la salida del modelo será una sola neurona sigmoide.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_example1(image, label):
    return tf.keras.applications.mobilenet_v2.preprocess_input(image), label

train_ds_1 = train_raw.map(preprocess_example1, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds_1   = val_raw.map(preprocess_example1, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_1  = test_raw.map(preprocess_example1, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

for imgs, labels in train_ds_1.take(1):
    print(f"Batch shape:  {imgs.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Rango de píxeles: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]")

---
## 1.3. Construir el modelo con Feature Extraction

1. Cargar MobileNetV2 **sin** el clasificador de ImageNet (`include_top=False`)
2. **Congelar** el backbone (`trainable = False`)
3. Agregar una cabeza binaria: `GlobalAveragePooling2D → Dropout → Dense(1, sigmoid)`

La salida es la probabilidad `P(Humano)`: si es ≥ 0.5 se clasifica como humano, en caso contrario como caballo.

In [ ]:
from tensorflow.keras import layers, models

base_model_1 = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE_1, 3)
)
base_model_1.trainable = False

print("Backbone: MobileNetV2")
print(f"Capas totales: {len(base_model_1.layers)}")
print(f"Parámetros del backbone: {base_model_1.count_params():,} (todos congelados)")
print(f"Output shape: {base_model_1.output_shape}")

In [ ]:
model_1 = models.Sequential([
    base_model_1,
    layers.GlobalAveragePooling2D(),          # 7x7x1280 -> 1280
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')     # P(Humano)
], name='horses_humans_mobilenetv2_FE')

model_1.summary()

total = model_1.count_params()
trainable = sum(tf.size(w).numpy() for w in model_1.trainable_weights)
print(f"\nTotal: {total:,} parámetros")
print(f"Entrenables: {trainable:,} ({trainable/total*100:.3f}%): 1280 pesos + 1 sesgo")
print(f"Congelados: {total - trainable:,} ({(total-trainable)/total*100:.2f}%)")

---
## 1.4. Entrenar y evaluar

In [ ]:
model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_1 = model_1.fit(
    train_ds_1,
    validation_data=val_ds_1,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

print(f"\nMejor val_accuracy: {max(history_1.history['val_accuracy']):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history_1.history['accuracy']) + 1)

axes[0].plot(epochs, history_1.history['accuracy'], 'o-', label='Train', color='#2196F3')
axes[0].plot(epochs, history_1.history['val_accuracy'], 's-', label='Val', color='#FF5722')
axes[0].set_title('Accuracy — Horses vs Humans (MobileNetV2 FE)', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_1.history['loss'], 'o-', label='Train', color='#2196F3')
axes[1].plot(epochs, history_1.history['val_loss'], 's-', label='Val', color='#FF5722')
axes[1].set_title('Loss — Horses vs Humans (MobileNetV2 FE)', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

### Evaluación final sobre el conjunto de test

El test no se usó para entrenar ni para elegir la mejor época.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_loss, test_acc = model_1.evaluate(test_ds_1, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}\n")

y_true = np.concatenate([y.numpy().ravel() for _, y in test_ds_1]).astype(int)
y_prob = model_1.predict(test_ds_1, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES_1, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES_1,
            yticklabels=CLASS_NAMES_1, ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicción', fontweight='bold'); ax.set_ylabel('Real', fontweight='bold')
ax.set_title('Matriz de confusión — Test', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 1.5. Predicciones visuales (conjunto de test)

In [ ]:
viz_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, shuffle=True, seed=SEED,
    image_size=IMG_SIZE_1, batch_size=10, label_mode='binary', class_names=FOLDERS)
imgs, labels = next(iter(viz_raw))

probs = model_1.predict(tf.keras.applications.mobilenet_v2.preprocess_input(imgs), verbose=0).ravel()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.ravel()):
    pred_idx = int(probs[i] >= 0.5)
    conf = probs[i] if pred_idx == 1 else 1 - probs[i]
    true_idx = int(labels[i].numpy()[0])
    ax.imshow(imgs[i].numpy().astype('uint8'))
    ax.set_title(f"Pred: {CLASS_NAMES_1[pred_idx]} ({conf:.0%})", fontsize=10,
                 color='green' if pred_idx == true_idx else 'red', fontweight='bold')
    ax.set_xlabel(f"Real: {CLASS_NAMES_1[true_idx]}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Predicciones — Feature Extraction (MobileNetV2) en Test', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Conclusión del Ejercicio 1

In [ ]:
best_val = max(history_1.history['val_accuracy'])
resumen = pd.DataFrame({
    'Métrica': ['Modelo base', 'Dataset', 'Clases', 'Parámetros entrenados',
                'Épocas ejecutadas', 'Mejor Val Accuracy', 'Test Accuracy'],
    'Valor': ['MobileNetV2 (ImageNet, congelado)', 'Horses or Humans (1,283 imgs)', '2 (binario)',
              f"{trainable:,} (solo la cabeza sigmoide)",
              len(history_1.history['accuracy']), f"{best_val:.4f}", f"{test_acc:.4f}"]
})
print(resumen.to_string(index=False))

El modelo entrena únicamente **1,281 parámetros** (la cabeza sigmoide). Los ~2.2M parámetros del backbone MobileNetV2 permanecen congelados y aportan los features aprendidos en ImageNet.